In [ ]:
# 1. install and start Ollama, then pull the models
!apt-get update -qq > /dev/null 2>&1; apt-get install -y -qq zstd > /dev/null 2>&1  # the installer needs zstd
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
# room for long prompts (Ollama silently cuts the start of a prompt longer than this); both models stay loaded
SERVER_ENV = dict(os.environ, OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_NUM_PARALLEL='3',
                  OLLAMA_MAX_LOADED_MODELS='2', OLLAMA_KEEP_ALIVE='24h')
# the notebook kernel refuses `!cmd &`, so the server starts as a detached subprocess
SERVER = subprocess.Popen('exec ollama serve > /tmp/ollama.log 2>&1', shell=True, env=SERVER_ENV,
                          start_new_session=True)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/ollama.log').read()[-2000:])
    raise SystemExit('ollama did not start; the log is above')
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b
# stop now, not hours later, if this GPU cannot run the models (Ollama would fall back to the CPU)
import json as _json
_req = urllib.request.Request('http://127.0.0.1:11434/api/generate', headers={'Content-Type': 'application/json'},
                              data=_json.dumps({'model': 'llama3.1:8b', 'prompt': 'Say ok.', 'stream': False}).encode())
urllib.request.urlopen(_req, timeout=900).read()
_ps = subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout
print(_ps)
if 'GPU' not in _ps:
    raise SystemExit('Ollama is running on the CPU here; use a supported GPU (T4)')

In [ ]:
# 2. unpack the project and the toolkit (outside /kaggle/working, so the saved output is only the tarball)
import base64, io, tarfile, os
BLOB = "..."
PROJ = '/tmp/rops-proj'
os.makedirs(PROJ, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode='r:gz') as t:
    t.extractall(PROJ)

In [ ]:
# 3. the grid and the report, with the same harness and rops.llm as locally
# (time-capped so the output is still written inside Kaggle's 12 h limit; a re-run resumes the grid)
!cd /tmp/rops-proj && ROPS_HOME=/tmp/rops-proj PYTHONPATH=/tmp/rops-proj ROPS_HTTP_TIMEOUT=600 timeout 39600 python3 -m rops.run grid /tmp/rops-proj --provider ollama
!cd /tmp/rops-proj && ROPS_HOME=/tmp/rops-proj PYTHONPATH=/tmp/rops-proj python3 -m rops.run report /tmp/rops-proj

In [ ]:
# 4. every analysis script
import glob, subprocess, sys
for script in sorted(glob.glob(PROJ + '/analysis/*.py')):
    done = subprocess.run([sys.executable, script], cwd=PROJ, env=dict(os.environ, PYTHONPATH=PROJ))
    print(script, 'exit', done.returncode)  # a failed analysis never blocks the output cell

In [ ]:
# 5. the output: runs/ and results/
import tarfile as _tar
with _tar.open('/kaggle/working/rops-output.tar.gz', 'w:gz') as out:
    for name in ('runs', 'results'):
        if os.path.isdir(os.path.join(PROJ, name)):
            out.add(os.path.join(PROJ, name), arcname=name)